# **트랜스포머와 LLM**


---

# **3. GPT를 직접 만들기**: 디코더 전용 트랜스포머

| Lab | 주제 | 출처 |
|---|---|---|
| 3-1 |  GPT의 학습 데이터 = 한 칸 밀린 정답 | Radford et al. (2018) 식 (1) |
| 3-2 |  mini-GPT 구현·학습 vs 9차시 문자 LSTM (같은 데이터·같은 스텝) | nanoGPT 구조 (Karpathy, 2023) |
| 3-3 |  디코딩 전략: greedy · temperature · top-k · top-p | Holtzman et al. (2020) |
| 3-4 |  GPT-2 124M 파라미터를 손으로 계산하고 확인 | Radford et al. (2019) |
| 3-5 |   KV 캐시가 생성 속도를 얼마나 바꾸는가 (실측) | Hugging Face `generate` |

In [1]:
%pip install -q koreanize-matplotlib "transformers>=4.51"

In [2]:
import os, math, time, random, json, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
os.makedirs("figs", exist_ok=True)
def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


---
## **3-1. GPT는 무엇을 배우는가?** — 다음 토큰 예측
$\mathcal{L} = -\sum_t \log P(x_t \mid x_{<t})$ — 라벨이 필요 없다. 문장 자체가 정답이다 (자기지도 학습).

In [3]:
seq = list("이 영화 최고")
print("입력 x :", seq[:-1])
print("정답 y :", seq[1:])
for t in range(1, len(seq)):
    print(f"  '{''.join(seq[:t])}' → '{seq[t]}'")
# 👉 길이 T인 문장 하나가 T−1개의 학습 문제가 된다. 인과 마스크 덕분에 이 T−1개를 한 번에(병렬로) 계산한다.

입력 x : ['이', ' ', '영', '화', ' ', '최']
정답 y : [' ', '영', '화', ' ', '최', '고']
  '이' → ' '
  '이 ' → '영'
  '이 영' → '화'
  '이 영화' → ' '
  '이 영화 ' → '최'
  '이 영화 최' → '고'


---
## **3-2. mini-GPT vs 문자 LSTM** — 같은 데이터, 같은 학습 스텝
데이터: NSMC 긍정 리뷰 (9차시 Lab 4-1과 동일한 전처리)

In [4]:
if not os.path.exists("ratings_train.txt"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", "ratings_train.txt")
nsmc = pd.read_csv("ratings_train.txt", sep="\t").dropna()
pos = [d.strip() for d in nsmc[nsmc.label == 1].document if 10 <= len(d.strip()) <= 80]
text = "\n".join(pos)
cnt = pd.Series(list(text)).value_counts()
chars = ["?"] + sorted(cnt[cnt >= 20].index)
c2i = {c: i for i, c in enumerate(chars)}
data = torch.tensor([c2i.get(c, 0) for c in text])
n_val = len(data) // 10
train_data, val_data = data[:-n_val], data[-n_val:]                  # 마지막 10%는 검증용
V = len(chars)
print(f"리뷰 {len(pos):,}개 | 문자 {len(data):,}개 | 어휘 {V:,}개 | 학습 {len(train_data):,} / 검증 {len(val_data):,}")

SEQ, BS, STEPS = 64, 64, 1000
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - SEQ - 1, (BS,))
    x = torch.stack([d[i:i+SEQ] for i in ix]); y = torch.stack([d[i+1:i+SEQ+1] for i in ix])
    return x.to(device), y.to(device)

리뷰 61,942개 | 문자 1,920,974개 | 어휘 1,145개 | 학습 1,728,877 / 검증 192,097


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.c_attn, self.c_proj, self.h = nn.Linear(C, 3 * C), nn.Linear(C, C), n_head   # GPT-2와 같은 이름
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)       # 인과 마스크: 미래 금지
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class GPTBlock(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.ln_1, self.attn, self.ln_2 = nn.LayerNorm(C), CausalSelfAttention(C, n_head), nn.LayerNorm(C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))      # Pre-LN + 잔차
        return x + self.mlp(self.ln_2(x))

class MiniGPT(nn.Module):
    def __init__(self, V, C=128, n_layer=4, n_head=4, block_size=SEQ):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, C), nn.Embedding(block_size, C)   # 토큰 임베딩 + 학습형 위치 임베딩
        self.blocks = nn.Sequential(*[GPTBlock(C, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(C)
        self.lm_head = nn.Linear(C, V, bias=False)
        self.lm_head.weight = self.wte.weight                                  # 가중치 공유 (weight tying)
        self.block_size = block_size
    def forward(self, idx):
        T = idx.size(1)
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        return self.lm_head(self.ln_f(self.blocks(x)))                          # (B, T, V) 다음 토큰 로짓

class CharLSTM(nn.Module):                                                      # 9차시 Lab 4-1과 같은 구조
    def __init__(self, V, emb=64, hidden=256):
        super().__init__()
        self.emb, self.lstm, self.fc = nn.Embedding(V, emb), nn.LSTM(emb, hidden, batch_first=True), nn.Linear(hidden, V)
        self.block_size = 10**9
    def forward(self, idx):
        return self.fc(self.lstm(self.emb(idx))[0])

@torch.no_grad()
def val_loss(m, n=20):
    m.eval(); seed_all(123)
    return float(np.mean([F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1)).item() for x, y in (get_batch("val") for _ in range(n))]))

def train(m, lr, warmup=0):
    m.to(device); seed_all(0)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) if warmup else 1.0)
    curve, t0 = [], time.time()
    for step in range(1, STEPS + 1):
        m.train(); x, y = get_batch("train")
        loss = F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
        if step % 200 == 0:
            curve.append(val_loss(m)); print(f"  step {step:>4}  val loss {curve[-1]:.3f}  ({time.time()-t0:.0f}s)")
    return curve, time.time() - t0

results, curves = {}, {}
for name, model, lr, wu in [("문자 LSTM (9차시)", CharLSTM(V), 3e-3, 0), ("mini-GPT (4층·4헤드·C=128)", MiniGPT(V), 3e-3, 100)]:
    print(name)
    curves[name], sec = train(model, lr, wu)
    results[name] = dict(model=model, params=sum(p.numel() for p in model.parameters()), val=curves[name][-1], sec=sec)

cmp_df = pd.DataFrame([[k, f'{v["params"]:,}', round(v["val"], 3), round(math.exp(v["val"]), 1), round(v["sec"])] for k, v in results.items()],
                      columns=["모델", "파라미터 수", "검증 손실", "퍼플렉서티 exp(손실)", "학습 시간(초)"])
cmp_df.to_csv("gpt_vs_lstm.csv", index=False)
cmp_df

문자 LSTM (9차시)
  step  200  val loss 3.484  (124s)
  step  400  val loss 3.218  (238s)
  step  600  val loss 3.099  (324s)
  step  800  val loss 3.031  (411s)


In [ ]:
plt.figure(figsize=(7, 4))
for (name, c), col in zip(curves.items(), ["#DC2626", "#004A80"]):
    plt.plot(range(200, STEPS + 1, 200), c, "o-", lw=2, c=col, label=name)
plt.xlabel("학습 스텝"); plt.ylabel("검증 손실 (교차 엔트로피)"); plt.title("같은 데이터·같은 스텝: 문자 LSTM vs mini-GPT")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("figs/gpt_vs_lstm.png", dpi=160); plt.show()

**관찰**: 1,000 스텝에서는 문자 LSTM이 앞선다. 같은 설정으로 4,000 스텝까지 늘린 추가 실험(CPU)에서도 검증 손실은 LSTM 2.823 · mini-GPT 2.905(lr 3e-3) · 2.975(lr 1e-3)로, 격차가 0.24 → 0.08로 줄었을 뿐 이 규모에서는 LSTM이 앞선다.
트랜스포머는 순서 관계를 데이터에서 배워야 해서 작은 모델·짧은 문맥에서는 불리하다. 우위는 **더 큰 모델 · 더 많은 데이터 · 긴 문맥 · GPU 병렬화**에서 나온다 (스케일링 법칙).
.

### [미션]: **`MiniGPT(V, C=192, n_layer=6, n_head=6)`, `SEQ=128`로 키우고 Colab GPU에서 STEPS=5000으로 다시 비교해 보자**

---
## **3-3. 디코딩 전략**: 같은 모델, 다른 문장
| 전략 | 규칙 |
|---|---|
| greedy | 항상 가장 높은 확률 1개 |
| temperature T | softmax(z/T) 후 샘플링 |
| top-k | 상위 k개만 남기고 샘플링 |
| top-p (nucleus) | 누적확률 p가 될 때까지의 최소 집합에서 샘플링 |

In [ ]:
@torch.no_grad()
def generate(m, prompt, n=60, strategy="sample", T=1.0, k=None, p=None):
    m.eval()
    idx = torch.tensor([[c2i.get(c, 0) for c in prompt]], device=device)
    for _ in range(n):
        logits = m(idx[:, -m.block_size:] if hasattr(m, "wpe") else idx)[:, -1] / (T if strategy != "greedy" else 1.0)
        if strategy == "greedy":
            nxt = logits.argmax(-1, keepdim=True)
        else:
            if k is not None:                                              # top-k: k번째보다 작은 로짓 제거
                kth = torch.topk(logits, k).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            probs = F.softmax(logits, -1)
            if p is not None:                                              # top-p: 누적확률 p 밖은 제거
                sp, si = torch.sort(probs, descending=True)
                remove = sp.cumsum(-1) - sp > p
                sp[remove] = 0
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
                probs = probs / probs.sum(-1, keepdim=True)
            nxt = torch.multinomial(probs, 1)
        if chars[nxt.item()] == "\n":
            break
        idx = torch.cat([idx, nxt], 1)
    return "".join(chars[i] for i in idx[0].tolist())

gpt = results["mini-GPT (4층·4헤드·C=128)"]["model"]
lstm = results["문자 LSTM (9차시)"]["model"]
rows = []
for label, kw in [("greedy", dict(strategy="greedy")), ("T=0.7", dict(T=0.7)), ("T=1.3", dict(T=1.3)),
                  ("top-k=10", dict(k=10)), ("top-p=0.9", dict(p=0.9))]:
    for prompt in ["이 영화", "배우들"]:
        seed_all(7)
        rows.append([label, prompt, generate(gpt, prompt, **kw)])
dec_df = pd.DataFrame(rows, columns=["전략", "시작어", "mini-GPT 생성 결과"])
dec_df.to_csv("decoding.csv", index=False)
dec_df

In [ ]:
seed_all(7); print("LSTM  (T=0.8):", generate(lstm, "이 영화", T=0.8))
seed_all(7); print("GPT   (T=0.8):", generate(gpt, "이 영화", T=0.8))

---
## **3-4. GPT-2** small(124M)의 파라미터를 손으로 계산하기
어휘 V=50,257 · 문맥 1,024 · C=768 · 12층 · 12헤드, 출력층은 입력 임베딩과 가중치 공유

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
V2, Tctx, C, L = 50257, 1024, 768, 12
by_hand = {"토큰 임베딩 V·C": V2 * C, "위치 임베딩 T·C": Tctx * C, "블록 12C²+13C × 12": L * (12 * C * C + 13 * C), "마지막 LayerNorm 2C": 2 * C}
total = sum(by_hand.values())
seed_all(0)
gpt2 = GPT2LMHeadModel(GPT2Config())            # 구조만 생성 (가중치는 랜덤, 다운로드 없음)
actual = sum(p.numel() for p in gpt2.parameters())
param_df = pd.DataFrame([[k, f"{v:,}", f"{v/total:.1%}"] for k, v in by_hand.items()] + [["합계 (손 계산)", f"{total:,}", "100%"], ["HF GPT2LMHeadModel 실제", f"{actual:,}", ""]],
                        columns=["구성", "파라미터 수", "비율"])
param_df.to_csv("gpt2_params.csv", index=False)
param_df

In [ ]:
print(gpt2.transformer.h[0])     # 블록 하나의 실제 구조: ln_1 → attn(c_attn, c_proj) → ln_2 → mlp(c_fc, c_proj)

---
## **3-5. KV 캐시**: 이미 계산한 K·V를 다시 계산하지 않기
캐시가 없으면 새 토큰 하나를 만들 때마다 **앞 문장 전체**의 K·V를 다시 계산한다. 캐시가 있으면 새 토큰의 K·V만 계산해 붙인다.

In [ ]:
gpt2.eval().to(device)
prompt_ids = torch.randint(0, V2, (1, 20), device=device)
rows = []
for n_new in [50, 100, 200]:
    for use_cache in [True, False]:
        torch.manual_seed(0)
        t0 = time.perf_counter()
        out = gpt2.generate(prompt_ids, max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False,
                            use_cache=use_cache, pad_token_id=0)
        rows.append([n_new, "캐시 사용" if use_cache else "캐시 없음", time.perf_counter() - t0])
kv = pd.DataFrame(rows, columns=["생성 토큰 수", "방식", "시간(초)"]).pivot(index="생성 토큰 수", columns="방식", values="시간(초)").round(2)
kv["속도 배수"] = (kv["캐시 없음"] / kv["캐시 사용"]).round(1)
kv.to_csv("kv_cache.csv")
kv

**관찰**: 생성 길이가 길수록 캐시의 효과가 커진다 (캐시 없음: 매 스텝 O(t) 재계산 → 전체 O(n²)).
대신 캐시는 메모리를 쓴다: 층 수 × 2(K,V) × KV헤드 수 × 헤드 차원 × 길이 — 그래서 최신 LLM은 GQA로 KV 헤드 수를 줄인다 .

## **정리**
| GPT의 요소 | 오늘 구현 |
|---|---|
| 디코더 전용 + 인과 마스크 | `is_causal=True` |
| 다음 토큰 예측 손실 | 입력을 한 칸 민 정답 |
| 가중치 공유 | `lm_head.weight = wte.weight` |
| 디코딩 전략 | greedy · temperature · top-k · top-p |
| KV 캐시 | `generate(use_cache=True)` |